# 🧱 LeoCAD Generative AI: Fine-Tuning Vision-Language Models (VLM)
### End-to-End Multimodal LDraw 3D Synthesizer on Google Colab (Free T4 GPU)

This notebook fine-tunes **Qwen2-VL-7B-Instruct** (or **LLaVA-1.5-7B**) to translate **arbitrary real-world reference photos + text prompts + target piece counts** into **executable 3D LEGO LDraw models** with step-by-step instructions.

- **Target Model**: Qwen2-VL-7B-Instruct with 4-bit QLoRA
- **Input**: Image + Text prompt (e.g., *"Build a red sports car with roughly 120 pieces"*)
- **Output**: Valid multi-step LDraw code block (`1 <color> <x> <y> <z> ... <part>.dat`)
- **Export Target**: Ollama Modelfile (`ollama run leocad-vlm`) connected directly to LeoCAD's AI Assistant.

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi

## 2. Install High-Performance VLM Fine-Tuning Libraries (Unsloth / PEFT / TRL)

In [ ]:
# Install Unsloth and Hugging Face dependencies optimized for fast 4-bit training
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers torchvision

## 3. Clone Repository or Upload LeoCAD Multimodal Dataset

In [ ]:
import os
if not os.path.exists("dataset"):
    !git clone https://github.com/Ulysse6/LeoCAD-Design-Agent.git leocad_repo
    !cp -r leocad_repo/dataset .
    !cp -r leocad_repo/models .

print("Checking dataset files...")
assert os.path.exists("dataset/training_pairs.jsonl"), "training_pairs.jsonl not found!"
with open("dataset/training_pairs.jsonl", "r") as f:
    sample_count = sum(1 for _ in f)
print(f"Found {sample_count} multimodal training pairs ready for fine-tuning.")

## 4. Load Base VLM Model with 4-bit Quantization (QLoRA)

In [ ]:
from unsloth import FastVisionModel
import torch

# Qwen2-VL-7B-Instruct 4-bit fits within ~6 GB of VRAM during LoRA training
model_name = "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=model_name,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# Apply LoRA parameter adapters
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,   # Adapt vision encoder to LEGO contours & parts
    finetune_language_layers=True, # Adapt LLM head to LDraw syntax
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
)

## 5. Prepare Conversational Multimodal Dataset

In [ ]:
import json
from PIL import Image

formatted_data = []
with open("dataset/training_pairs.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        img_path = entry.get("image", "")
        if not img_path or not os.path.exists(img_path):
            continue
        
        user_text = entry["conversations"][0]["value"].replace("<image>\n", "").strip()
        assistant_text = entry["conversations"][1]["value"].strip()
        
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": user_text}
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": assistant_text}
            ]}
        ]
        formatted_data.append(messages)

print(f"Loaded {len(formatted_data)} formatted multimodal chat conversations.")

## 6. Execute LoRA Fine-Tuning

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=15,
    max_steps=120,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir="leocad_vlm_lora",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=None,
    train_dataset=formatted_data,
    args=training_args,
)

trainer.train()

## 7. Inference Test: Synthesizing an Arbitrary Novel LEGO Model

In [ ]:
# Run test generation with FastVisionModel inference mode
FastVisionModel.for_inference(model)

test_prompt = "Build an arbitrary yellow speeder car with roughly 80 pieces and step-by-step assembly."
test_img = Image.open("dataset/images/model_001_car_side.png")

messages = [
    {"role": "user", "content": [
        {"type": "image", "image": test_img},
        {"type": "text", "text": test_prompt}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(test_img, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=512, use_cache=True)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

## 8. Export to Ollama for LeoCAD Integration

In [ ]:
# Save LoRA adapter and merged GGUF weights
model.save_pretrained("leocad_vlm_lora")
tokenizer.save_pretrained("leocad_vlm_lora")

# Create Ollama Modelfile
modelfile_content = """FROM qwen2-vl:7b
ADAPTER ./leocad_vlm_lora
PARAMETER temperature 0.3
PARAMETER top_p 0.9
SYSTEM \""You are the LeoCAD 3D LEGO Design Agent. When given an image and a user request with a target piece budget, output valid LDraw format code containing '1 <colour> <x> <y> <z> <a> <b> <c> <d> <e> <f> <g> <h> <i> <part>.dat' commands organized into steps with '0 STEP' comments.\""
"""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile created. In your local terminal with Ollama installed, run:")
print("  ollama create leocad-vlm -f Modelfile")
print("  ollama run leocad-vlm")
print("Then select 'Neural VLM (Ollama / Local)' in LeoCAD's AI Assistant!")